#Paquetes necesarios

In [1]:
import cv2  
import math 

from ultralytics import YOLO

Modelos preentrenados, visualizando con las utilidades de ultralytics

In [2]:
# Carga del modelo
#model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolo11n-seg.pt') #Máscaras
model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model(filename, show=True)

cv2.destroyAllWindows()


WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 79.6ms
video 1/1 (frame 2/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 53.4ms
video 1/1 (frame 3/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 53.9ms
video 1/1 (frame 4/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 54.6ms

Desde cámara, detección con yolo11, modelo nano. Visualización propia con OpenCV

In [6]:
# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') #Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck", "boat",
              "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
              "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
              "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat",
              "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
              "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
              "carrot", "hot dog", "pizza", "donut", "cake", "chair", "sofa", "pottedplant", "bed",
              "diningtable", "toilet", "tvmonitor", "laptop", "mouse", "remote", "keyboard", "cell phone",
              "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
              "teddy bear", "hair drier", "toothbrush"
              ]


# Captura desde la webcam
vid = cv2.VideoCapture(0)
  
while(True):      
    # fotograma a fotograma
    ret, img = vid.read()
  
    # si hay imagen válida
    if ret:  
        # Detecta en la imagen
        results = model(img, stream=True)
        
        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) # convert to int values
                
                # Confianza
                confidence = math.ceil((box.conf[0]*100))/100
                print("Confianza --->",confidence)

                # Clase
                cls = int(box.cls[0])
                print("Clase -->", classNames[cls])

                # Convierte identificador numérico de clase a un color RGB
                escala = int((cls / len(classNames)) * 255 * 3)
                if escala >= 255*2:
                    R = 255
                    G = 255
                    B = escala - 255*2
                else:
                    if escala >= 255:
                        R = 255
                        G = escala - 255
                        B = 0
                    else:
                        R = escala
                        G = 0
                        B = 0

                # Dibuja el contenedor y clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

        # Muestra fotograma
        cv2.imshow('Vid', img)
    
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
# Destruye ventanas
cv2.destroyAllWindows()


0: 480x640 (no detections), 60.4ms
Speed: 1.3ms preprocess, 60.4ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 60.6ms
Speed: 1.1ms preprocess, 60.6ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 60.6ms
Speed: 1.1ms preprocess, 60.6ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 58.6ms
Speed: 1.5ms preprocess, 58.6ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 57.6ms
Speed: 1.0ms preprocess, 57.6ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 60.0ms
Speed: 1.6ms preprocess, 60.0ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 63.0ms
Speed: 1.6ms preprocess, 63.0ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 66.0ms
Speed: 1.1ms preprocess, 66.0ms i

Seguimiento. Requiere instalar lap con pip install lap

In [2]:
from collections import defaultdict
import numpy as np

# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') #Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car", "plate"]


# Captura desde la webcam
vid = cv2.VideoCapture(0)
track_history = defaultdict(lambda: [])
  
while(True):      
    # fotograma a fotograma
    ret, img = vid.read()
  
    # si hay imagen válida
    if ret:  
        # Seguimiento, con persistencia entre fotogramas
        results = model.track(img, persist=True, classes = [0,1,2])

        if 0:
            if results is not None:
                print(results[0])
                boxes = results[0].boxes.xywh.cpu()
                track_ids = results[0].boxes.id.int().cpu().tolist()
                annotated_frame = results[0].plot()
                for box, track_id in zip(boxes, track_ids):
                    x, y, w, h = box
                    track = track_history[track_id]
                    track.append((float(x), float(y)))
                    if len(track) > 30:
                        track.pop(0)
                    points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                    cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 230, 230), thickness=10)
                cv2.imshow("YOLO11 Tracking", annotated_frame)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
        

        
        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) # convert to int values

                #Etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0].tolist()))
                else:
                    track_id = ''
                
                # Confianza
                confidence = math.ceil((box.conf[0]*100))/100
                print("Confianza --->",confidence)

                # Clase
                cls = int(box.cls[0])
                print("Clase -->", classNames[cls])

                # Convierte identificador numérico de clase a un color RGB
                escala = int((cls / len(classNames)) * 255 * 3)
                if escala >= 255*2:
                    R = 255
                    G = 255
                    B = escala - 255*2
                else:
                    if escala >= 255:
                        R = 255
                        G = escala - 255
                        B = 0
                    else:
                        R = escala
                        G = 0
                        B = 0

                # Dibuja el contenedor y clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, track_id + ' ' + classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

        # Muestra fotograma
        cv2.imshow('Vid', img)
    
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
# Destruye ventanas
cv2.destroyAllWindows()




0: 480x640 (no detections), 188.4ms
Speed: 3.0ms preprocess, 188.4ms inference, 3.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 18.4ms
Speed: 2.2ms preprocess, 18.4ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 17.8ms
Speed: 2.1ms preprocess, 17.8ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 21.2ms
Speed: 1.8ms preprocess, 21.2ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 20.4ms
Speed: 1.9ms preprocess, 20.4ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 13.2ms
Speed: 1.8ms preprocess, 13.2ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 31.6ms
Speed: 3.5ms preprocess, 31.6ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 27.3ms
Speed: 2.3ms preprocess, 27.3ms

Intregración con seguimiento (tracking)
!!!!!!!!!Nota: he tenido que bajar a la versión de python 3.9.5 e instalar lap con pip install lap

In [4]:
# Carga del modelo
model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolov11n-seg.pt') #Máscaras
#model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model.track(source=filename, show=True)  # BoT-SORT tracker (por defecto)
#results = model.track(source=filename, show=True, tracker="bytetrack.yaml")  # ByteTrack tracker

cv2.destroyAllWindows()


WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 3 persons, 59.5ms
video 1/1 (frame 2/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 3 persons, 55.1ms
video 1/1 (frame 3/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 3 persons, 53.3ms
video 1/1 (frame 4/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 3 persons, 52.2ms
video 1/1 (frame 5/375)

In [ ]:

data_path = r"C:\Users\luisp\Desktop\VC\prac1\P4\matriculas\data\matriculas.yaml"
model = YOLO("yolo11n.pt")

results = model.train(
    data=data_path,
    epochs=180,
    imgsz=1080,
    batch=-1,
    device=0,
    amp=True,
    rect=True,
    workers=4,

    optimizer="adamw",
    cos_lr=True,
    lr0=8e-4,
    lrf=1e-2,
    patience=30,

    mosaic=0.25,
    close_mosaic=15,
    mixup=0.0,
    copy_paste=0.0,
    fliplr=0.0,
    degrees=3.0,
    translate=0.05,
    scale=0.50,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,

    save_period=10,
    project="runs/detect",
    name="plates_s_1280_rect",
    exist_ok=True,
)


New https://pypi.org/project/ultralytics/8.3.225 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.223  Python-3.9.24 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\luisp\Desktop\VC\prac1\P4\matriculas\data\matriculas.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=180, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=1080, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0008, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.25, multi_

In [4]:
import os, csv, cv2
from collections import defaultdict, Counter, deque
from ultralytics import YOLO

# =========================
# RUTAS (AJUSTA)
# =========================
VIDEO_IN  = r"C:\Users\luisp\Desktop\VC\prac1\P4\videos\video_matriculas.mp4"
VIDEO_OUT = r"C:\Users\luisp\Desktop\VC\prac1\P4\outputs\video_annotado.mp4"
CSV_OUT   = r"C:\Users\luisp\Desktop\VC\prac1\P4\outputs\detecciones.csv"

# =========================
# MODELOS
# =========================
detector = YOLO("yolo11n.pt")  # personas/vehículos (COCO)
plate_model = YOLO(r"C:\Users\luisp\Desktop\VC\prac1\P4\runs\detect\plates_s_1280_rect\weights\best.pt")  # tu modelo de matrículas

# =========================
# OPCIONES DETECCIÓN / TRACKING
# =========================
TARGET_CLASSES = {"person", "car", "motorbike", "bus", "truck"}
TRACKER = "bytetrack.yaml"  # o un yaml propio (p.ej. "bytetrack_strict.yaml")
DET_CONF = 0.25

# Matrículas
PLATE_CONF = 0.45           # sube para reducir FPs
PLATE_IOU = 0.50
PLATE_IMGSZ = 960           # subir a 1024–1280 para placas lejanas
PLATE_ONLY_BOTTOM_BAND = True
BOTTOM_FRAC = 0.40          # 40% inferior del vehículo
EXTRA_BAND_UP = 0.05        # +5% hacia arriba

# Filtros geométricos EU (placa 1 línea)
PLATE_AR_MIN, PLATE_AR_MAX = 3.8, 5.8   # relación de aspecto típica EU 1 línea
ALLOW_TWO_LINE = False                   # True para permitir AR 1.2–2.0 (2 líneas)
MIN_PLATE_AREA = 900                     # píxeles mínimos

# Relativos al vehículo
VEH_PLATE_AREA_FRAC_MIN = 0.003         # >=0.3% del área del vehículo
VEH_PLATE_AREA_FRAC_MAX = 0.06          # <=6%
PLATE_VH_FRAC_MIN = 0.06                # h_placa >= 6% h_vehículo
PLATE_VH_FRAC_MAX = 0.25                # h_placa <= 25% h_vehículo

# Extras
ANONYMIZE = False               # difumina personas/vehículos
USE_CONTOUR_FALLBACK = True     # contornos si no detecta YOLO
ANTI_HEADLIGHT = True           # filtro simple anti-faros
FLOW_ANALYSIS = True            # flujo direccional

# Histeresis anti-parpadeo por track
HIST_N = 5                      # ventana
ACCEPT_K = 3                    # aceptar si >=K detecciones válidas en la ventana
HOLD_FRAMES = 8                 # mantener última buena N frames si falla 1-2 frames

# Debounce de salidas
MISSING_TOLERANCE = 12          # frames ausente antes de contar salida (~0.5s a 25 fps)

# =========================
# UTILIDADES
# =========================
def clamp_roi(x1, y1, x2, y2, W, H):
    x1 = max(0, min(W, x1)); x2 = max(0, min(W, x2))
    y1 = max(0, min(H, y1)); y2 = max(0, min(H, y2))
    return x1, y1, x2, y2

def plausible_plate_absrel(w, h, veh_w, veh_h):
    area = w * h
    if area < MIN_PLATE_AREA:
        return False
    ar = w / max(1, h)
    if ALLOW_TWO_LINE and (1.2 <= ar <= 2.0):
        pass
    elif not (PLATE_AR_MIN <= ar <= PLATE_AR_MAX):
        return False
    vw = max(1, veh_w); vh = max(1, veh_h)
    rel_area = area / (vw * vh)
    if not (VEH_PLATE_AREA_FRAC_MIN <= rel_area <= VEH_PLATE_AREA_FRAC_MAX):
        return False
    rel_h = h / vh
    if not (PLATE_VH_FRAC_MIN <= rel_h <= PLATE_VH_FRAC_MAX):
        return False
    return True

def blur_box(img, x1, y1, x2, y2, k=31):
    x1,y1,x2,y2 = map(int, (x1,y1,x2,y2))
    roi = img[y1:y2, x1:x2]
    if roi.size == 0: 
        return img
    blur = cv2.GaussianBlur(roi, (k|1, k|1), 0)
    img[y1:y2, x1:x2] = blur
    return img

def find_plate_by_contours(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 9, 75, 75)
    edges = cv2.Canny(gray, 50, 150)
    edges = cv2.dilate(edges, None, iterations=1)
    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best, best_area = None, 0
    Hc, Wc = crop.shape[:2]
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        if w < 20 or h < 10: 
            continue
        if x < 2 or y < 2 or x+w > Wc-2 or y+h > Hc-2:
            continue
        if w*h > best_area:
            best = (x, y, x+w, y+h); best_area = w*h
    return best

def is_headlight_like(crop_box):
    g = cv2.cvtColor(crop_box, cv2.COLOR_BGR2GRAY)
    thr = cv2.threshold(g, 235, 255, cv2.THRESH_BINARY)[1]
    white_frac = thr.mean() / 255.0
    if white_frac > 0.80:  # muy plano/blanco
        return True
    edges = cv2.Canny(g, 80, 160)
    vert = cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=3)
    if edges.mean() < 5 and abs(vert).mean() < 2:  # poca estructura
        return True
    return False

# Histeresis por track
plate_history = {}    # tid -> deque[(mx1,my1,mx2,my2,conf) or None]
plate_last_good = {}  # tid -> (mx1,my1,mx2,my2,conf)
plate_hold = {}       # tid -> frames restantes de hold

def push_history(tid, box_conf):
    dq = plate_history.setdefault(tid, deque(maxlen=HIST_N))
    dq.append(box_conf)

def accept_by_majority(tid):
    dq = plate_history.get(tid, [])
    return sum(1 for b in dq if b is not None) >= ACCEPT_K

def use_last_good_if_holding(tid):
    if plate_hold.get(tid, 0) > 0 and tid in plate_last_good:
        plate_hold[tid] -= 1
        return plate_last_good[tid]
    return None

# Debounce de salidas
inactive_counter = {}   # id -> frames ausente consecutivos
already_counted = set() # ids ya contados como salida

# =========================
# IO
# =========================
os.makedirs(os.path.dirname(VIDEO_OUT), exist_ok=True)
os.makedirs(os.path.dirname(CSV_OUT), exist_ok=True)

cap = cv2.VideoCapture(VIDEO_IN)
if not cap.isOpened():
    raise FileNotFoundError(f"No puedo abrir el vídeo: {VIDEO_IN}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(VIDEO_OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

csv_f = open(CSV_OUT, "w", newline="", encoding="utf-8")
cw = csv.writer(csv_f)
cw.writerow([
    "fotograma","tipo_objeto","confianza","identificador_tracking","x1","y1","x2","y2",
    "matrícula_en_su_caso","confianza_matricula","mx1","my1","mx2","my2","texto_matricula"
])

seen_ids_by_class = defaultdict(set)
last_centroid = {}
exit_side_count = Counter()

frame_idx = 0

# =========================
# LOOP
# =========================
while True:
    ok, frame = cap.read()
    if not ok:
        break

    gen = detector.track(
        source=frame, stream=True, persist=True,
        tracker=TRACKER, conf=DET_CONF, verbose=False
    )
    try:
        res = next(gen)
    except StopIteration:
        res = None

    if res is None or res.boxes is None or len(res.boxes) == 0:
        writer.write(frame)
        frame_idx += 1
        continue

    names = detector.model.names
    boxes = res.boxes
    active_ids = set()

    for b in boxes:
        if b.cls is None or b.conf is None or b.xyxy is None:
            continue

        cls_id = int(b.cls[0].item())
        conf   = float(b.conf[0].item())
        name   = names.get(cls_id, str(cls_id))
        if name not in TARGET_CLASSES:
            continue

        x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
        tid = int(b.id[0].item()) if b.id is not None else -1
        active_ids.add(tid)

        if tid != -1:
            seen_ids_by_class[name].add(tid)

        cx = int((x1 + x2) * 0.5)
        cy = int((y1 + y2) * 0.5)
        last_centroid[tid] = (cx, cy)

        if ANONYMIZE:
            frame = blur_box(frame, x1, y1, x2, y2, k=35)
        else:
            color = (0, 255, 0) if name != "person" else (0, 200, 255)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"{name} {conf:.2f} ID:{tid}",
                        (x1, max(0, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        # ---------- MATRÍCULAS SOLO EN VEHÍCULOS ----------
        plate_flag, plate_conf, (mx1,my1,mx2,my2), plate_text = 0, 0.0, (0,0,0,0), ""
        if name in {"car","motorbike","bus","truck"}:
            vx1, vy1, vx2, vy2 = x1, y1, x2, y2
            if PLATE_ONLY_BOTTOM_BAND:
                vh = vy2 - vy1
                band_top = vy2 - int(BOTTOM_FRAC * vh)
                band_top = band_top - int(EXTRA_BAND_UP * vh)
                rx1, ry1, rx2, ry2 = vx1, band_top, vx2, vy2
            else:
                rx1, ry1, rx2, ry2 = vx1, vy1, vx2, vy2

            rx1, ry1, rx2, ry2 = clamp_roi(rx1, ry1, rx2, ry2, W, H)
            if rx2 > rx1 and ry2 > ry1:
                crop = frame[ry1:ry2, rx1:rx2]

                # 1) YOLO de placas
                pp = plate_model.predict(
                    source=crop, conf=PLATE_CONF, iou=PLATE_IOU,
                    imgsz=PLATE_IMGSZ, max_det=3, verbose=False
                )

                candidate, pbest = None, 0.0
                if pp and len(pp[0].boxes) > 0:
                    for pb in pp[0].boxes:
                        px1, py1, px2, py2 = map(int, pb.xyxy[0].tolist())
                        pconf = float(pb.conf[0].item())
                        wpl, hpl = px2 - px1, py2 - py1
                        if pconf > pbest:
                            candidate = (px1, py1, px2, py2, pconf)
                            pbest = pconf

                # 2) Fallback por contornos si YOLO no ve nada
                if candidate is None and USE_CONTOUR_FALLBACK:
                    cbb = find_plate_by_contours(crop)
                    if cbb is not None:
                        px1, py1, px2, py2 = cbb
                        candidate = (px1, py1, px2, py2, 0.30)

                tid_key = tid
                if candidate is not None:
                    px1, py1, px2, py2, pconf = candidate
                    mx1 = rx1 + px1
                    my1 = ry1 + py1
                    mx2 = rx1 + px2
                    my2 = ry1 + py2

                    wpl, hpl = mx2 - mx1, my2 - my1
                    veh_w, veh_h = vx2 - vx1, vy2 - vy1

                    # Anti-faro (opcional)
                    if ANTI_HEADLIGHT:
                        plate_crop = frame[my1:my2, mx1:mx2]
                        if plate_crop.size != 0 and is_headlight_like(plate_crop):
                            push_history(tid_key, None)
                        else:
                            if plausible_plate_absrel(wpl, hpl, veh_w, veh_h):
                                push_history(tid_key, (mx1,my1,mx2,my2,pconf))
                            else:
                                push_history(tid_key, None)
                    else:
                        if plausible_plate_absrel(wpl, hpl, veh_w, veh_h):
                            push_history(tid_key, (mx1,my1,mx2,my2,pconf))
                        else:
                            push_history(tid_key, None)
                else:
                    push_history(tid_key, None)

                # Histeresis (aceptación/hold)
                g = None
                if accept_by_majority(tid_key):
                    # fija última buena
                    # usa la más reciente no-None de la ventana
                    dq = plate_history.get(tid_key, [])
                    last = next((bc for bc in reversed(dq) if bc is not None), None)
                    if last is not None:
                        plate_last_good[tid_key] = last
                        plate_hold[tid_key] = HOLD_FRAMES
                        g = last
                else:
                    g = use_last_good_if_holding(tid_key)

                if g:
                    gx1, gy1, gx2, gy2, gc = g
                    plate_flag, plate_conf = 1, gc
                    if not ANONYMIZE:
                        cv2.rectangle(frame, (gx1, gy1), (gx2, gy2), (255, 0, 0), 2)
                        cv2.putText(frame, f"PLATE {gc:.2f}",
                                    (gx1, max(0, gy1 - 6)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

        # CSV por detección
        cw.writerow([
            frame_idx, name, f"{conf:.3f}", tid, x1, y1, x2, y2,
            plate_flag, f"{plate_conf:.3f}", mx1, my1, mx2, my2, ""
        ])

    # --------- Debounce de salidas (flujo) ---------
    if FLOW_ANALYSIS:
        # incrementa inactivos y cuenta salida si supera tolerancia
        ids_to_check = set(list(last_centroid.keys()) + list(inactive_counter.keys()))
        for gid in list(ids_to_check):
            if gid in active_ids:
                inactive_counter[gid] = 0
            else:
                inactive_counter[gid] = inactive_counter.get(gid, 0) + 1
                if inactive_counter[gid] == MISSING_TOLERANCE and gid not in already_counted:
                    cx, cy = last_centroid.get(gid, (None, None))
                    if cx is not None:
                        dists = {"left": cx, "right": W - cx, "top": cy, "bottom": H - cy}
                        side = min(dists, key=dists.get)
                        exit_side_count[side] += 1
                        already_counted.add(gid)
                    last_centroid.pop(gid, None)

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()
csv_f.close()

# =========================
# RESÚMENES
# =========================
print("\nIDs únicos por clase:")
for k, ids in seen_ids_by_class.items():
    ids.discard(-1)
    print(f"  {k}: {len(ids)}")

if FLOW_ANALYSIS:
    print("\nFlujo (salidas por borde):")
    for k in ("left","right","top","bottom"):
        print(f"  {k}: {exit_side_count[k]}")

print(f"\nVídeo anotado: {VIDEO_OUT}")
print(f"CSV: {CSV_OUT}")



IDs únicos por clase:
  car: 319
  truck: 151
  bus: 40
  person: 8

Flujo (salidas por borde):
  left: 73
  right: 1
  top: 163
  bottom: 159

Vídeo anotado: C:\Users\luisp\Desktop\VC\prac1\P4\outputs\video_annotado.mp4
CSV: C:\Users\luisp\Desktop\VC\prac1\P4\outputs\detecciones.csv
